In [1]:
cd Prosody2Vec/

/home/dcor/niskhizov/Prosody2Vec


In [2]:
import torch

In [3]:
from torch import nn
import torch 
import glob
from IPython.display import clear_output, display, Audio
import copy

from utils import reset_model_weights

In [4]:
import torch

In [5]:
import model

In [6]:
# data_dir = './Emotion Speech Dataset/'
# data_dir = '/home/dcor/niskhizov/Prosody2Vec/IEMOCAP_full_release/'
data_dir = '/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/'
# scan recursively for all .wav files in the data_dir
wav_files = glob.glob(data_dir + '/**/*.wav', recursive=True)



In [7]:
len(wav_files)

35000

In [8]:
# embeddings_dir = 'esd_female_018'
embeddings_dir = './esd_3sec_embeddings'

In [9]:
# create pytorch dataset that loads pairs of wav a and embeddings from iemocap_embeddings
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import pickle
import torchaudio

class IemocapDataset(Dataset):
    def __init__(self, audio_files):
        self.audio_files = []
        self.embeddings_file = []

        for audio_file in audio_files:
            out_file = f"{embeddings_dir}/{audio_file.split('/')[-1].replace('.wav', '.pkl')}"
            if os.path.exists(out_file):                
                self.embeddings_file.append(out_file)
                self.audio_files.append(audio_file)

    def __len__(self):
        return len(self.embeddings_file)
    
    def __getitem__(self, idx):

        wav_path = self.audio_files[idx]

        out_file = self.embeddings_file[idx]

        with open(out_file, 'rb') as f:
            embd = pickle.load(f)

        wav,sr = torchaudio.load(wav_path)

        # take the first 3 seconds of the audio

        wav = wav[:, :3*sr]

        
        
        return wav, embd

In [10]:
from speechbrain.inference.speaker import EncoderClassifier

spk_ecapa_tdnn = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
spk_ecapa_tdnn.device = 'cuda'

reset_model_weights(spk_ecapa_tdnn)
spk_ecapa_tdnn.mods.embedding_model.blocks[0].conv.conv.weight.requires_grad == True


/home/dcor/niskhizov/anaconda3/lib/python3.12/site-packages/speechbrain/utils/autocast.py:68: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
/home/dcor/niskhizov/anaconda3/lib/python3.12/site-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly 

Model weights reinitialized and requires_grad set to True!


True

In [11]:


class FusionDecoderV4(nn.Module):
    def __init__(self, hidden_dim, acoustic, spk_ecapa_tdnn):
        super(FusionDecoderV4, self).__init__()
        # self.attn = AttentionFusion(prosody_dim, hidden_dim)
        
        self.ff1 = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 128))
        self.ff2 = nn.Sequential(nn.Linear(512+256, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 512))
        self.ff3 = nn.Sequential(nn.Linear(192, 128), nn.ReLU(), nn.Linear(128, 128))

        self.base_model = copy.deepcopy(acoustic)

        self.ecapa = spk_ecapa_tdnn


    def forward(self, units, wav, spk_vecs, logmels):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)
        # logmels: (batch_size, time, n_mels)
        
        # batch_size, time, _ = units.shape
        emo_vecs = self.ecapa.encode_batch(wav.cuda()).squeeze(1)

        # Apply attention
        o = self.base_model.encoder(units.cuda())

        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))

        d = self.base_model.decoder(o3, logmels)

        return d
    
    def generate(self, units, wav, spk_vecs):
        # units: (batch_size, time, hidden_dim)
        # emo_vecs: (batch_size, emo_vec_size)
        # spk_vecs: (batch_size, spk_vec_size)

        emo_vecs = self.ecapa.encode_batch(wav.cuda()).squeeze(1)
        # Apply attention
        o = self.base_model.encoder(units.cuda())

        # units_attn = self.attn(o, emo_vecs)
        o2 = self.ff1(emo_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)
        o2b = self.ff3(spk_vecs.cuda()).unsqueeze(1).expand(-1,o.shape[1] , -1)

        # units_attn = self.attn(o, emo_vecs)  # (batch_size, time, hidden_dim)

        o3 = self.ff2(torch.cat([o, o2, o2b], dim=-1))


        d = self.base_model.decoder.generate(o3)
        
        return d


In [12]:
import time

In [13]:
wav_files_english = [x for x in wav_files if int(x.split('/')[-3]) > 10] 

In [14]:
ds = IemocapDataset(wav_files_english)

In [15]:
train_ds, test_ds = torch.utils.data.random_split(ds, [int(0.8*len(ds)), len(ds) - int(0.8*len(ds))])

In [16]:
# create collate function that will pad the sequences to the same length
def collate_fn(batch):
    wavs = [item[0][0] for item in batch]
    
    d_units, units, emo_vecs, spk_vecs, logmels = [], [], [], [], []
    for item in batch:
        d = item[1]['discrite_units']
        u = item[1]['units']
        mel = item[1]['logmel'].T

        d_units.append(d)
        units.append(u)
        emo_vecs.append(torch.tensor((item[1]['emo_vec'])))
        spk_vecs.append(item[1]['spk_vec'])

        mel  = mel[:u.size(0)*2,:]
        # print(mel.shape)
        mel = torch.nn.functional.pad(mel, (0,0,1,0))
        # print(mel.shape)

        logmels.append(mel)

    
    mels_lengths = torch.tensor([x.size(0) - 1 for x in logmels])
    units_lengths = torch.tensor([x.size(0) for x in units])

    d_units_padded = nn.utils.rnn.pad_sequence(d_units, batch_first=True, padding_value=-1)
    units_padded = nn.utils.rnn.pad_sequence(units, batch_first=True)
    logmels_padded = nn.utils.rnn.pad_sequence(logmels, batch_first=True)
    
    _,T,_ = units_padded.shape
    # pad the sequences

    wavs = nn.utils.rnn.pad_sequence(wavs, batch_first=True)

    
    return wavs, d_units_padded, units_padded, torch.stack(emo_vecs), torch.stack(spk_vecs), logmels_padded, mels_lengths, units_lengths

In [17]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=10)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=10)

In [18]:
acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()
decoder = FusionDecoderV4(512, acoustic, spk_ecapa_tdnn).cuda()


Using cache found in /home/dcor/niskhizov/cache/hub/bshall_acoustic-model_main


In [19]:
from torch.optim import Adam
from torch.nn.functional import l1_loss

optimizer = Adam(decoder.parameters(), lr=1e-4)


In [20]:
from tqdm import tqdm_notebook,tqdm

In [ ]:
for epoch in range(0,300000):  
    decoder.train()
    for idx,batch in tqdm(enumerate(train_dl),total=len(train_dl)):
        wavs, d_units_padded, units_padded, emo_vecs, spk_vecs, logmels_padded, mels_lengths, units_lengths  = batch
        
        optimizer.zero_grad()

        out =  decoder(d_units_padded.cuda(),wavs.cuda(),spk_vecs,logmels_padded[:, 1:, :].cuda())
        # out = decoder(d_units_padded.cuda(), emo_vecs.cuda(), spk_vecs.cuda(), logmels_padded[:, :-1, :].cuda())
        # out = acoustic(units_padded.cuda(), logmels_padded[:, :-1, :].cuda())

        # target = hifigan(out[:1,:,:].transpose(1, 2))
        loss = l1_loss(out, logmels_padded[:, 1:, :].cuda(), reduction="none")
        loss = torch.sum(loss, dim=(1, 2)) / (out.size(-1) * mels_lengths.cuda())
        loss = torch.mean(loss)
        loss.backward()

        optimizer.step()

    if epoch % 500 == 0:
        print('Epoch:', epoch, 'Batch:', idx)
        print('Loss:', loss.item())

    if epoch % 500 == 0:
        torch.save(decoder.state_dict(), f"decoder_pretraining_multi_speaker_{epoch}.pth")

100%|██████████| 69/69 [00:44<00:00,  1.57it/s]


Epoch: 0 Batch: 68
Loss: 0.5264102220535278


  0%|          | 0/69 [00:00<?, ?it/s]

In [ ]:
epoch

In [ ]:
import plotly.express as px


In [ ]:
px.imshow(out[0].detach().cpu().numpy().T)

## Inference

In [ ]:
ls -lash --sort time | grep decoder

In [ ]:
# load the latest decoder
decoders = glob.glob('decoder_*.pth')
# sort by last modification time
decoders.sort(key=os.path.getmtime)
decoder.load_state_dict(torch.load(decoders[-1]))
print(decoders[-1])
decoder.eval()


In [ ]:
it = iter(test_dl)

In [ ]:
batch = next(it)

In [ ]:
hifigan = torch.hub.load("bshall/hifigan:main", "hifigan_hubert_discrete", trust_repo=True).cuda()

In [ ]:
# with torch.no_grad():
        
#         cont_units = decoder.decoder_rnn.encoder(d_units.cuda())

#         units_attn = decoder.attn(cont_units.cuda(), emo_vecs.cuda(), spk_vecs.cuda())  # (batch_size, time, hidden_dim)
        
   
#         o = decoder.decoder_rnn.decoder.generate(units_attn)
wavs,d_units, units_padded, emo_vecs, spk_vecs, logmels_padded, mels_lengths, units_lengths  = batch

decoder = decoder.eval()

with torch.no_grad():
        

        o = decoder.generate(d_units[0].unsqueeze(0).cuda(), emo_vecs[0].unsqueeze(0).cuda(),spk_vecs[0].unsqueeze(0).cuda())
        

In [ ]:
spk_vecs[12].shape

In [ ]:
# acoustic = torch.hub.load("bshall/acoustic-model:main", "hubert_discrete", trust_repo=True).cuda()

# with torch.no_grad():
#     o = acoustic.generate(d_units.cuda())

In [ ]:
import plotly.express as px
px.imshow(o[0].detach().cpu().numpy().T)

In [ ]:
idx = 0
with torch.no_grad():
    target = hifigan(o[idx,:,:].unsqueeze(0).transpose(1, 2))

In [ ]:
Audio(target[0].detach().cpu().numpy(), rate=16000)

In [ ]:
Audio(wavs[0],rate=16000)

In [ ]:
Audio(wavs[12],rate=16000)

In [ ]:
hubert_discrete = torch.hub.load("bshall/hubert:main", "hubert_discrete", trust_repo=True).cuda()


In [ ]:
from funasr import AutoModel


In [ ]:
model_id = "iic/emotion2vec_plus_large"

sed_model = AutoModel(
    model=model_id,
    hub="ms",  # "ms" or "modelscope" for China mainland users; "hf" or "huggingface" for other overseas users
)

In [ ]:
def extract_embedding(wav_path):
    wav, sr = torchaudio.load(wav_path)

    # take 3 seconds of audio

    with torch.inference_mode():
        # Extract speech units
        discrite_units = hubert_discrete.units(wav.unsqueeze(0).cuda())
        
        emo_vec = torch.tensor(sed_model.generate(wav, granularity="utterance", extract_embedding=True, disable_pbar =True)[0]['feats'])

    return discrite_units, emo_vec, wav



In [ ]:
neutral_wavs = glob.glob('Emotion Speech Dataset/0018/Neutral/*.wav')

In [ ]:
angry_wavs = glob.glob('Emotion Speech Dataset/0018/Angry/*.wav')

In [ ]:
happy_wavs = glob.glob('Emotion Speech Dataset/0018/Happy/*.wav')

In [ ]:
wav_a = "/home/dcor/niskhizov/Prosody2Vec/Emotion Speech Dataset/0018/Sad/0018_001305.wav"
wav_b = angry_wavs[0]
wav_c = happy_wavs[0]
embed_a = extract_embedding(wav_a)
embed_b = extract_embedding(wav_b)
embed_c = extract_embedding(wav_c)


In [ ]:

decoder = decoder.eval()

with torch.no_grad():
        

        o = decoder.generate(embed_a[0].unsqueeze(0).cuda(), embed_b[1].unsqueeze(0).cuda())
        

In [ ]:
with torch.no_grad():
    target = hifigan(o.transpose(1, 2)).cpu()[0][0]

In [ ]:
Audio(target,rate = 16000)

In [ ]:
Audio(embed_a[-1],rate = 16000)

In [ ]:
Audio(embed_b[-1],rate = 16000)

In [ ]:
with torch.no_grad():
        

        o = decoder.generate(embed_a[0].unsqueeze(0).cuda(), embed_c[1].unsqueeze(0).cuda())
        

In [ ]:
with torch.no_grad():
    target = hifigan(o.transpose(1, 2)).cpu()[0][0]

In [ ]:
Audio(target,rate = 16000)


In [ ]:
Audio(embed_a[-1],rate = 16000)

In [ ]:
Audio(embed_b[-1],rate = 16000)